In [1]:
import keras
from keras import layers
from keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
# b. Access the dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
# Convert 28x28 images into 784 values
x_train = x_train.reshape((len(x_train), 784))
x_test = x_test.reshape((len(x_test), 784))
print("Training data:", x_train.shape)
print("Testing data :", x_test.shape)

C:\Users\parde\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.8). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


Training data: (60000, 784)
Testing data : (10000, 784)


In [2]:
# =========================================================
# c. Encoder - converts input into latent representation
# =========================================================
encoding_dim = 32
input_img = keras.Input(shape=(784,))
encoded = layers.Dense(
128, activation="relu"
)(input_img)
encoded = layers.Dense(
encoding_dim, activation="relu"
)(encoded)
# =========================================================
# d. Decoder - converts latent representation back
# to original input
# =========================================================
decoded = layers.Dense(
128, activation="relu"
)(encoded)
decoded = layers.Dense(
784, activation="sigmoid"
)(decoded)
# Create Autoencoder
autoencoder = keras.Model(input_img, decoded)
# =========================================================
# e. Compile model
# =========================================================
autoencoder.compile(
optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"]
)
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           4,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │           4,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 784)                 │         101,136 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 209,968 (820.19 KB)

 Trainable params: 209,968 (820.19 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# =========================================================
# Train the Autoencoder
# =========================================================
autoencoder.fit(
x_train,
x_train,
epochs=10,
batch_size=256,
shuffle=True,
validation_data=(x_test, x_test)
)

Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.0111 - loss: 0.2271 - val_accuracy: 0.0100 - val_loss: 0.1475
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0100 - loss: 0.1324 - val_accuracy: 0.0095 - val_loss: 0.1193
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0104 - loss: 0.1152 - val_accuracy: 0.0116 - val_loss: 0.1086
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0111 - loss: 0.1070 - val_accuracy: 0.0128 - val_loss: 0.1029
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0112 - loss: 0.1025 - val_accuracy: 0.0128 - val_loss: 0.0997
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0108 - loss: 0.0997 - val_accuracy: 0.0111 - val_loss: 0.0974
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0110 - loss: 0.0978 - val_accuracy: 0.0123 - val_loss: 0.0957
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0118 - loss: 0.0962 - val_accuracy: 0.

In [4]:
# Anomaly Detection
import numpy as np
import matplotlib.pyplot as plt
# Reconstruct test images
decoded_imgs = autoencoder.predict(x_test)
# Calculate reconstruction error
error = np.mean(
    np.square(x_test - decoded_imgs),
    axis=1
)
# Set threshold
threshold = np.percentile(error, 95)
print("Anomaly Threshold:", threshold)
# Detect anomalies
anomalies = error > threshold
print("Number of anomalies:", np.sum(anomalies))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Anomaly Threshold: 0.021012576
Number of anomalies: 500
